# 14_btc_correlation_analysis — Bitcoin 历史相关性与风险收益分析

> 重新审视: BTC 是否真的不该进 V5C? 用数据回答, 不用直觉。

## 分析维度

1. **全周期相关性矩阵** (BTC 自 2014-09)
2. **滚动 1 年相关性** (检测 regime change)
3. **分时期相关性** (5 个 regime: 2014-2018 / 2018-2020 / 2020-2022 / 2022-2024 / 2024-2026)
4. **危机期表现** (2018-Q4 / 2020 COVID / 2022 Bear / 2025 Q1)
5. **单标的风险收益** (CAGR / Vol / Sharpe / Max DD)
6. **假设场景**: V5C 3.3a + BTC 1%/3%/5% 的影响

## 核心问题

- BTC 与各资产相关性如何随时间演变?
- BTC 在 2022 滞胀期是 hedge 还是跟跌?
- BTC 的 11Y 风险调整后回报到底有多高?
- 如果加 1-5% BTC 到 V5C 3.3a, Sharpe 改善还是恶化?

## 警告

- BTC 11Y 数据是 outlier 时期 (从极小市值崛起)
- 历史 CAGR 不代表未来
- 但相关性 + 危机表现是有效信息

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

tickers = {
    'BTC': 'BTC-USD',
    'QQQ': 'QQQ',
    'VOO': 'VFINX',
    'TLT': 'TLT',
    'VGSH': 'VFITX',
    'GLDM': 'GLD',
    'BCX': 'DBC',
}

raw = yf.download(list(tickers.values()), start='2014-01-01', auto_adjust=True)['Close']
rename = {v: k for k, v in tickers.items()}
raw.columns = [rename.get(c, c) for c in raw.columns]

print('数据起始:')
for k in tickers:
    if k in raw.columns:
        first = raw[k].first_valid_index()
        print(f'  {k}: {first.date() if first else "N/A"}')

data = raw.dropna()
returns = data.pct_change().dropna()
print(f'\n共同期间: {data.index[0].date()} → {data.index[-1].date()} ({len(data)/252:.1f} 年)')

In [ ]:
# ============================================================
# 1. 全周期相关性矩阵
# ============================================================
print('='*70)
print('1. 全周期日收益率相关性矩阵 (~11.7 年)')
print('='*70)

corr_full = returns.corr()
print('\n相关性矩阵:')
print(corr_full.round(3))

# 单看 BTC 与其他
print('\nBTC 与各资产的相关性 (按高到低):')
btc_corr = corr_full['BTC'].drop('BTC').sort_values(ascending=False)
for k, v in btc_corr.items():
    sign = '+' if v >= 0 else ''
    print(f'  BTC vs {k:<6}: {sign}{v:.3f}')

# 热力图
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr_full, annot=True, fmt='.2f', cmap='RdBu_r', center=0, 
            vmin=-1, vmax=1, square=True, cbar_kws={'shrink': 0.8})
ax.set_title(f'全周期日收益相关性 ({data.index[0].date()} - {data.index[-1].date()})')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 2. BTC 的滚动 1 年相关性 (regime change 检测)
# ============================================================
window = 252
rolling_corr = pd.DataFrame(index=returns.index)
for col in ['QQQ','VOO','TLT','VGSH','GLDM','BCX']:
    rolling_corr[f'BTC_vs_{col}'] = returns['BTC'].rolling(window).corr(returns[col])

fig, ax = plt.subplots(figsize=(15, 7))
for col in rolling_corr.columns:
    ax.plot(rolling_corr.index, rolling_corr[col].values, label=col.replace('BTC_vs_', 'vs '), alpha=0.85, linewidth=1.5)
ax.axhline(0, color='black', linewidth=0.5)
ax.axhline(0.3, color='gray', linewidth=0.5, linestyle='--', alpha=0.5)
ax.axhline(-0.3, color='gray', linewidth=0.5, linestyle='--', alpha=0.5)
ax.set_title('BTC 滚动 1 年相关性 (检测 regime change)', fontsize=14)
ax.set_ylabel('Correlation')
ax.legend(loc='upper left', ncol=3)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# 关键时点的相关性
print('\nBTC 在关键时点的滚动 1Y 相关性:')
key_dates = ['2018-12-31','2020-03-23','2021-12-31','2022-12-31','2024-12-31','2026-05-01']
for d in key_dates:
    nearest_idx = rolling_corr.index.get_indexer([pd.Timestamp(d)], method='nearest')[0]
    actual_d = rolling_corr.index[nearest_idx]
    row = rolling_corr.iloc[nearest_idx]
    print(f'\n{actual_d.date()}:')
    for col, val in row.items():
        if pd.notna(val):
            print(f'  {col}: {val:+.3f}')

In [ ]:
# ============================================================
# 3. 分时期相关性 (5 个 regime)
# ============================================================
print('='*82)
print('3. BTC 在 5 个时代的相关性 (各资产 vs BTC)')
print('='*82)

regimes = {
    '2014-2017 早期 (零售为主)':  ('2014-09-17', '2017-12-31'),
    '2018-2019 加密寒冬':         ('2018-01-01', '2019-12-31'),
    '2020-2021 COVID + 加密牛':   ('2020-01-01', '2021-12-31'),
    '2022-2023 加密寒冬 + 通胀':   ('2022-01-01', '2023-12-31'),
    '2024-2026 ETF + 机构化':      ('2024-01-01', '2026-05-31'),
}

asset_list = ['QQQ','VOO','TLT','VGSH','GLDM','BCX']
regime_corr = pd.DataFrame(index=asset_list, columns=list(regimes.keys()))

for regime_name, (s, e) in regimes.items():
    sub = returns.loc[s:e]
    if len(sub) < 60:
        continue
    for asset in asset_list:
        regime_corr.loc[asset, regime_name] = sub['BTC'].corr(sub[asset])

regime_corr = regime_corr.astype(float)
print('\nBTC vs 各资产 在 5 个时代的相关性:')
print(regime_corr.round(3))

# 热力图
fig, ax = plt.subplots(figsize=(13, 5))
sns.heatmap(regime_corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, 
            vmin=-0.6, vmax=0.6, cbar_kws={'shrink': 0.8})
ax.set_title('BTC 与各资产相关性的时代演化')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 4. 危机期表现 (核心测试: BTC 在 stress 下是 hedge 还是跟跌?)
# ============================================================
print('='*82)
print('4. 危机期单标的表现 (BTC 行为模式核心验证)')
print('='*82)

crises = {
    '2018-Q4 跌势':         ('2018-10-01', '2018-12-31'),
    '2020 COVID 急跌':      ('2020-02-19', '2020-04-30'),
    '2020 流动性极端':      ('2020-03-09', '2020-03-23'),
    '2022 Bear 全年':       ('2022-01-01', '2022-12-31'),
    '2022 滞胀核心':        ('2022-01-01', '2022-09-30'),
    '2025 Q1 关税':         ('2025-01-01', '2025-04-30'),
}

asset_order = ['BTC','QQQ','VOO','TLT','VGSH','GLDM','BCX']
print(f'\n{"危机":<22}', end='')
for a in asset_order: print(f' {a:>10}', end='')
print()
print('-'*100)
for n, (s, e) in crises.items():
    if pd.Timestamp(s) < returns.index[0]: continue
    line = f'{n:<22}'
    for a in asset_order:
        r = (1 + returns[a].loc[s:e]).prod() - 1
        line += f' {r:>+9.2%}'
    print(line)

In [ ]:
# ============================================================
# 5. 单标的风险收益指标 (全周期, ~11.7 年)
# ============================================================
print('='*70)
print('5. 单标的风险收益指标 (~11.7 年)')
print('='*70)

results = []
for asset in asset_order:
    r = returns[asset]
    cum = (1+r).cumprod()
    n_y = len(r)/252
    cagr = cum.iloc[-1]**(1/n_y) - 1
    vol = r.std()*np.sqrt(252)
    sharpe = (cagr - 0.04) / vol
    rm = cum.expanding().max()
    dd = (cum/rm - 1).min()
    calmar = cagr/abs(dd)
    results.append({
        'Asset': asset,
        'CAGR': cagr,
        'Vol': vol,
        'Sharpe': sharpe,
        'Max DD': dd,
        'Calmar': calmar,
    })

df = pd.DataFrame(results).set_index('Asset')
df['CAGR'] = df['CAGR'].apply(lambda x: f'{x:+.2%}')
df['Vol'] = df['Vol'].apply(lambda x: f'{x:.2%}')
df['Sharpe'] = df['Sharpe'].apply(lambda x: f'{x:+.3f}')
df['Max DD'] = df['Max DD'].apply(lambda x: f'{x:.2%}')
df['Calmar'] = df['Calmar'].apply(lambda x: f'{x:.3f}')
print(df)

# 净值曲线
fig, ax = plt.subplots(figsize=(14, 7))
for asset in asset_order:
    cum = (1 + returns[asset]).cumprod()
    ax.plot(cum, label=f'{asset} (CAGR {(cum.iloc[-1]**(252/len(returns))-1):.0%})', linewidth=1.8, alpha=0.85)
ax.set_title(f'各资产净值曲线 ({data.index[0].date()} - {data.index[-1].date()}, log scale)', fontsize=13)
ax.set_yscale('log')
ax.legend(loc='upper left')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 6. 假设场景: V5C 3.3a + BTC 1% / 3% / 5%
# ============================================================
print('='*82)
print('6. V5C 3.3a 加入 BTC 后的影响 (~11.7 年)')
print('='*82)

# 注意: V5C 3.3a 的完整组合需要 HQH/XLV/DBMF, 但这里聚焦 BTC 影响
# 简化为: 用 VOO/QQQ/TLT/VGSH/GLDM/BCX 当近似 V5C

# 先用 6 资产模拟一个简化 V5C 3.3a (按比例缩放)
# 进攻: VOO 10 + QQQ 10 = 20% (HQH/XLV 10+10 用 VOO 替代, 总 40)
# 实际比例: VOO 30, QQQ 10 (HQH/XLV 合并到 VOO)
# 对冲: GLDM 20 + BCX 10 + TLT 10 (DBMF 用 TLT 替代)
# 防御: VGSH 20

V5C_simplified = {'VOO':0.30, 'QQQ':0.10, 'GLDM':0.20, 'BCX':0.10, 'TLT':0.10, 'VGSH':0.20}

def simulate_no_rebal(returns_df, weights):
    sub = returns_df[list(weights.keys())]
    target = np.array(list(weights.values()))
    target = target / target.sum()
    cw = target.copy()
    pr = []
    rd = [sub.index[0]]
    for date, dr in sub.iterrows():
        pr.append(np.sum(cw*dr.values))
        nw = cw*(1+dr.values); nw = nw/nw.sum()
        if np.max(np.abs(nw - target))*100 >= 5.0:
            cw = target.copy(); rd.append(date)
        else:
            cw = nw
    return pd.Series(pr, index=sub.index), rd

def metrics(rs, name):
    cum = (1+rs).cumprod()
    n_y = len(rs)/252
    cagr = cum.iloc[-1]**(1/n_y) - 1
    vol = rs.std()*np.sqrt(252)
    sharpe = (cagr-0.04)/vol
    rm = cum.expanding().max()
    dd = (cum/rm - 1).min()
    return {'Name':name,'CAGR':cagr,'Vol':vol,'Sharpe':sharpe,'Max DD':dd,'Calmar':cagr/abs(dd)}

configs = {
    'V5C base (无 BTC)': V5C_simplified,
    'V5C + BTC 1%': {**{k:v*0.99 for k,v in V5C_simplified.items()}, 'BTC':0.01},
    'V5C + BTC 3%': {**{k:v*0.97 for k,v in V5C_simplified.items()}, 'BTC':0.03},
    'V5C + BTC 5%': {**{k:v*0.95 for k,v in V5C_simplified.items()}, 'BTC':0.05},
}

all_results = []
for name, w in configs.items():
    r, _ = simulate_no_rebal(returns, w)
    all_results.append(metrics(r, name))

df = pd.DataFrame(all_results).set_index('Name')
df['CAGR'] = df['CAGR'].apply(lambda x: f'{x:+.2%}')
df['Vol'] = df['Vol'].apply(lambda x: f'{x:.2%}')
df['Sharpe'] = df['Sharpe'].apply(lambda x: f'{x:+.3f}')
df['Max DD'] = df['Max DD'].apply(lambda x: f'{x:.2%}')
df['Calmar'] = df['Calmar'].apply(lambda x: f'{x:.3f}')
print(df)

print('\n说明:')
print('- 这是简化版 V5C 3.3a (用 VOO 30 替代 HQH/XLV, TLT 替代 DBMF)')
print('- 主要看 BTC 不同权重对 Sharpe / Max DD 的影响')
print('- 由于 BTC 11Y CAGR 极高(60-80%), 加入即使 1% 也会显著拉高 CAGR')
print('- 但 Max DD 会加深 (BTC 自身 -77% 历史回撤)')

## 关键判断标准

看完上面 6 项分析，判断 BTC 是否值得加入 V5C 3.3a:

**值得考虑加入 (1-3%)** 如果:
- 全周期 vs VOO 相关性 < 0.3 (低相关)
- 5 个时代相关性都不超过 0.5 (规则一致)
- 危机期表现至少有时是 hedge (不全部跟跌)
- 加入 BTC 后组合 Sharpe 显著提升

**应当回避** 如果:
- 近期 (2024-2026) 相关性 > 0.5 (机构化后变成 NDX 杠杆版)
- 危机期一致跟跌 (2020 COVID, 2022 Bear)
- 加入 BTC 后 Sharpe 不显著提升或下降
- Max DD 大幅恶化 (单 BTC -77% 拖累)

## 最重要的认知

无论结果如何, BTC 11Y 的极端 CAGR (估计 60-80%) **不会重复**:
- 当时市值小 (从几十亿 → 万亿)
- 早期采纳红利已经被 capture
- 未来的 "alpha" 不在过去的 11 年, 在下一个未知周期

所以即使 BTC 历史数据看起来好, 也要清醒:
**这是 outlier 数据 + 不可重复的早期采纳红利**.

决定不基于 "过去 BTC 涨了", 而基于 **"BTC 在未来组合中扮演什么角色"**.